# ELECTRA + ScalarMix + DANN — Shrishti train + OneStop (judge) → test CoQA + RACE

**Train** (`education_level_judge` everywhere):
- Shrishti `train.csv`
- OneStop from `ood_onestop.csv` (judge labels — consistent with Shrishti)

**Not in train:** CoQA (test only)

**Validation:**  `val.csv` (judge labels, DANN domains)

**OOD test:** full `coqa_train.csv`, `ood_race-middle.csv`, `ood_race-high.csv` (judge gold)

Two-phase DANN: frozen encoder → partial unfreeze + GRL. No vLLM required.

**Checkpoints saved**
- `best_model.pt` — best  macro-F1 (often phase 1)
- `phase2_final.pt` — weights after phase 2 (DANN), 

The final eval cell reports **both** on CoQA + RACE 

In [ ]:
!pip install -q transformers scikit-learn torch pandas matplotlib seaborn tqdm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# ── Paths (edit for your Drive) ───────────────────────────────────────────
DRIVE_CLEAN_DIR = "/content/drive/MyDrive/BeyondFK/clean_dataset"
COQA_TEST_CSV = "/content/drive/MyDrive/BeyondFK/trail/judge_coqa_cnn/clean_dataset/coqa_train.csv"
DRIVE_OUT_DIR = "/content/drive/MyDrive/BeyondFK/trail/electra_mixed_shrishti_onestop_dann"

MODEL_NAME = "google/electra-large-discriminator"
TEXT_COL = "full_text"
LABEL_COL = "education_level_judge"
SOURCE_COL = "source_dataset"

MAX_LEN = 512
BATCH_SIZE = 4
RNG_SEED = 42
LABEL_SMOOTHING = 0.1
MIN_CHARS = 80
DEDUPE_TEXT = True
EVAL_LABELS = [0, 1, 2]

USE_SHRISHTI_TRAIN = True
USE_ONESTOP_IN_TRAIN = True

# DANN
GRL_LAMBDA_MAX = 0.15
DOMAIN_LOSS_ALPHA = 0.1
PHASE1_EPOCHS = 2
PHASE1_LR = 1e-3
PHASE2_EPOCHS = 1
PHASE2_LR = 2e-5
PARTIAL_FREEZE_LAYERS = 8
WARMUP_STEPS = 100

label2id = {"elementary": 0, "middle": 1, "high": 2}
id2label = {v: k for k, v in label2id.items()}

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import random
from pathlib import Path
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from torch import Tensor
from torch.nn import Parameter, ParameterList
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

CLEAN_DIR = Path(DRIVE_CLEAN_DIR)
for p, name in [
    (CLEAN_DIR / "train.csv", "Shrishti train"),
    (CLEAN_DIR / "val.csv", "Shrishti val"),
    (CLEAN_DIR / "ood_onestop.csv", "OneStop"),
    (CLEAN_DIR / "ood_race-middle.csv", "RACE-middle"),
    (CLEAN_DIR / "ood_race-high.csv", "RACE-high"),
    (Path(COQA_TEST_CSV), "CoQA test"),
]:
    if not p.exists():
        raise FileNotFoundError(f"Missing {name}: {p}")

os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
CM_DIR = os.path.join(DRIVE_OUT_DIR, "confusion_matrices")
os.makedirs(CM_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(RNG_SEED)
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RNG_SEED)
print("Device:", device)

In [ ]:
# Shared model definitions live in DomainAdversialNN/dann_models.py
import sys
from pathlib import Path

for module_dir in [
    Path.cwd(),
    Path.cwd() / "DomainAdversialNN",
    Path("/content/drive/MyDrive/BeyondFlesch/Beyond-Flesch/DomainAdversialNN"),
    Path("/content/drive/MyDrive/BeyondFK/DomainAdversialNN"),
]:
    if module_dir.exists() and str(module_dir) not in sys.path:
        sys.path.append(str(module_dir))

from dann_models import (
    ElectraScalarMixDANN,
    build_domain2id,
    combined_loss,
    grl_lambda_schedule,
)

print("Shared DANN model imports OK")

In [ ]:
def _text_key(text: str) -> str:
    return hashlib.sha256(str(text).strip().encode("utf-8")).hexdigest()


def rows_from_csv(path: Path, pool: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if TEXT_COL not in df.columns or LABEL_COL not in df.columns:
        raise ValueError(f"{path}: need {TEXT_COL} and {LABEL_COL}")
    out = pd.DataFrame()
    out[TEXT_COL] = df[TEXT_COL].astype(str)
    out["label_str"] = df[LABEL_COL].astype(str).str.strip().str.lower()
    out["pool"] = pool
    if SOURCE_COL in df.columns:
        out[SOURCE_COL] = df[SOURCE_COL].astype(str)
    else:
        out[SOURCE_COL] = pool.replace("_train", "").replace("_test", "")
    out = out[out[TEXT_COL].str.len() >= MIN_CHARS].copy()
    bad = ~out["label_str"].isin(label2id)
    if bad.any():
        print(f"  [{pool}] drop {bad.sum()} unknown labels")
        out = out[~bad]
    out["label_id"] = out["label_str"].map(label2id).astype(int)
    return out.reset_index(drop=True)


def dedupe_pools(df: pd.DataFrame, priority: List[str]) -> pd.DataFrame:
    if not DEDUPE_TEXT:
        return df
    rank = {p: i for i, p in enumerate(priority)}
    df = df.copy()
    df["_rk"] = df["pool"].map(lambda p: rank.get(p, 999))
    df["_key"] = df[TEXT_COL].map(_text_key)
    df = df.sort_values("_rk").drop_duplicates("_key", keep="first")
    return df.drop(columns=["_rk", "_key"]).reset_index(drop=True)


def assign_domain_ids(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    out = df.copy()
    out["domain_id"] = out[SOURCE_COL].map(domain2id)
    unk = out["domain_id"].isna()
    if unk.any():
        print(f"  [{split_name}] drop {unk.sum()} rows with unknown {SOURCE_COL}")
        out = out[~unk].reset_index(drop=True)
    out["domain_id"] = out["domain_id"].astype(int)
    return out


parts = []
if USE_SHRISHTI_TRAIN:
    parts.append(rows_from_csv(CLEAN_DIR / "train.csv", "shrishti_train"))
if USE_ONESTOP_IN_TRAIN:
    parts.append(rows_from_csv(CLEAN_DIR / "ood_onestop.csv", "onestop_train"))

df_train = pd.concat(parts, ignore_index=True)
df_train = dedupe_pools(df_train, priority=["shrishti_train", "onestop_train"])
df_val = rows_from_csv(CLEAN_DIR / "val.csv", "shrishti_val")

domain2id, num_domains = build_domain2id(
    pd.concat([df_train[SOURCE_COL], df_val[SOURCE_COL]]).tolist()
)
print(f"num_domains={num_domains}: {list(domain2id.keys())}")

df_train = assign_domain_ids(df_train, "train")
df_val = assign_domain_ids(df_val, "val")

ood_eval = {
    "coqa_test": rows_from_csv(Path(COQA_TEST_CSV), "coqa_test"),
    "ood_race-middle": rows_from_csv(CLEAN_DIR / "ood_race-middle.csv", "race-middle"),
    "ood_race-high": rows_from_csv(CLEAN_DIR / "ood_race-high.csv", "race-high"),
}

print(f"Train rows: {len(df_train)}")
print(df_train.groupby(["pool", "label_str"]).size().unstack(fill_value=0))
print(f"\nTrain domains:\n", df_train[SOURCE_COL].value_counts())
print(f"\nVal rows: {len(df_val)} | labels: {dict(df_val['label_str'].value_counts())}")
for name, odf in ood_eval.items():
    print(f"\nOOD {name}: n={len(odf)} | gold: {dict(odf['label_str'].value_counts())}")

manifest = {
    "label_col": LABEL_COL,
    "train_pools": df_train["pool"].value_counts().to_dict(),
    "train_domains": df_train[SOURCE_COL].value_counts().to_dict(),
    "coqa_in_train": False,
    "domain2id": domain2id,
}
with open(os.path.join(DRIVE_OUT_DIR, "data_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class TextClsDataset(Dataset):
    def __init__(self, texts: List[str], labels: np.ndarray) -> None:
        self.texts = texts
        self.labels = labels.astype(int)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        enc = tokenizer(
            self.texts[idx], truncation=True, max_length=MAX_LEN,
            padding="max_length", return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


class TextDANNDataset(Dataset):
    def __init__(self, texts: List[str], labels: np.ndarray, domains: np.ndarray) -> None:
        self.texts = texts
        self.labels = labels.astype(int)
        self.domains = domains.astype(int)

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        enc = tokenizer(
            self.texts[idx], truncation=True, max_length=MAX_LEN,
            padding="max_length", return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
            "domain": torch.tensor(self.domains[idx], dtype=torch.long),
        }


def df_to_dann_loader(df: pd.DataFrame, shuffle: bool) -> DataLoader:
    return DataLoader(
        TextDANNDataset(df[TEXT_COL].tolist(), df["label_id"].values, df["domain_id"].values),
        batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0,
    )

train_loader = df_to_dann_loader(df_train, shuffle=True)
val_loader = df_to_dann_loader(df_val, shuffle=False)

In [ ]:
model = ElectraScalarMixDANN(MODEL_NAME, num_classes=3, num_domains=num_domains).to(device)
best_path = os.path.join(DRIVE_OUT_DIR, "best_model.pt")  # best Shrishti val macro-F1
phase2_final_path = os.path.join(DRIVE_OUT_DIR, "phase2_final.pt")
history: List[Dict] = []
best_val_f1 = -1.0
global_step = 0
total_steps = (PHASE1_EPOCHS + PHASE2_EPOCHS) * len(train_loader)


@torch.no_grad()
def evaluate_val(grl_lambda: float = 0.0) -> Dict[str, float]:
    model.eval()
    ys, preds, ds, dp = [], [], [], []
    for batch in val_loader:
        ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        diff_logits, dom_logits = model(ids, mask, grl_lambda=grl_lambda)
        ys.extend(batch["label"].numpy().tolist())
        preds.extend(diff_logits.argmax(dim=1).cpu().numpy().tolist())
        ds.extend(batch["domain"].numpy().tolist())
        dp.extend(dom_logits.argmax(dim=1).cpu().numpy().tolist())
    return {
        "val_macro_f1": float(f1_score(ys, preds, labels=EVAL_LABELS, average="macro", zero_division=0)),
        "val_acc": float(accuracy_score(ys, preds)),
        "val_domain_acc": float(accuracy_score(ds, dp)),
    }


def run_phase(phase_name: str, epochs: int, lr: float, grl_on: bool) -> None:
    global global_step, best_val_f1
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=min(WARMUP_STEPS, max(1, len(train_loader))),
        num_training_steps=max(1, epochs * len(train_loader)),
    )
    for epoch in range(epochs):
        model.train()
        run_diff, run_dom, run_total, n_batches = 0.0, 0.0, 0.0, 0
        grl_l = 0.0
        for batch in tqdm(train_loader, desc=f"{phase_name} ep{epoch+1}/{epochs}"):
            progress = global_step / max(total_steps - 1, 1)
            grl_l = grl_lambda_schedule(progress, max_lambda=GRL_LAMBDA_MAX) if grl_on else 0.0
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            domains = batch["domain"].to(device)
            diff_logits, dom_logits = model(ids, mask, grl_lambda=grl_l)
            total, diff_loss, dom_loss = combined_loss(
                diff_logits,
                dom_logits,
                labels,
                domains,
                domain_loss_alpha=DOMAIN_LOSS_ALPHA,
                label_smoothing=LABEL_SMOOTHING,
            )
            optimizer.zero_grad()
            total.backward()
            optimizer.step()
            scheduler.step()
            global_step += 1
            run_diff += diff_loss.item()
            run_dom += dom_loss.item()
            run_total += total.item()
            n_batches += 1
        metrics = evaluate_val(grl_lambda=0.0)
        rec = {
            "phase": phase_name, "epoch": epoch + 1, "grl_on": grl_on,
            "grl_lambda": grl_l, "train_diff_loss": run_diff / max(n_batches, 1),
            "train_dom_loss": run_dom / max(n_batches, 1),
            "train_total_loss": run_total / max(n_batches, 1), **metrics,
        }
        history.append(rec)
        print(
            f"{phase_name} ep{epoch+1}: diff={rec['train_diff_loss']:.4f} dom={rec['train_dom_loss']:.4f} "
            f"val_f1={rec['val_macro_f1']:.4f} val_dom_acc={rec['val_domain_acc']:.4f}"
        )
        if metrics["val_macro_f1"] > best_val_f1:
            best_val_f1 = metrics["val_macro_f1"]
            torch.save({"state_dict": model.state_dict(), "domain2id": domain2id}, best_path)
            print(f"  saved best (val macro-F1 {best_val_f1:.4f})")


model.freeze_encoder()
run_phase("phase1_frozen_encoder", PHASE1_EPOCHS, PHASE1_LR, grl_on=False)

model.unfreeze_encoder()
model.freeze_encoder_except_top(PARTIAL_FREEZE_LAYERS)
run_phase("phase2_dann", PHASE2_EPOCHS, PHASE2_LR, grl_on=True)

torch.save(
    {"state_dict": model.state_dict(), "domain2id": domain2id, "phase": "phase2_dann_final"},
    phase2_final_path,
)
print(f"Saved phase-2 final weights → {phase2_final_path}")
print(f"Best val macro-F1 checkpoint → {best_path} (F1={best_val_f1:.4f})")
print("Run the next cell to compare both checkpoints on CoQA + RACE.")

In [ ]:
def save_confusion_matrix(y_true, y_pred, title, out_path, labels):
    names = [id2label[i] for i in labels]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=names, yticklabels=names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True (judge)")
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


def load_checkpoint(path: str) -> None:
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    print(f"Loaded {path} (phase={ckpt.get('phase', 'best_val')})")


@torch.no_grad()
def predict_texts(texts: List[str], batch_size: int = 16) -> np.ndarray:
    model.eval()
    preds = []
    for i in tqdm(range(0, len(texts), batch_size), desc="predict", leave=False):
        enc = tokenizer(
            texts[i : i + batch_size], truncation=True, max_length=MAX_LEN,
            padding=True, return_tensors="pt",
        )
        logits = model.difficulty_logits_only(enc["input_ids"].to(device), enc["attention_mask"].to(device))
        preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
    return np.array(preds, dtype=int)


def eval_split(
    name: str, y_true: np.ndarray, y_pred: np.ndarray,
    checkpoint_tag: str, verbose: bool = True,
) -> Dict:
    if len(y_true) == 0:
        raise ValueError(f"{name}: no samples")
    labels = EVAL_LABELS
    present = sorted(set(y_true.tolist()) | set(y_pred.tolist()))
    acc = accuracy_score(y_true, y_pred)
    f1_all = f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)
    f1_present = f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)
    if verbose:
        report = classification_report(
            y_true, y_pred, labels=labels,
            target_names=[id2label[i] for i in labels], zero_division=0,
        )
        print(f"\n{'='*60}\n[{checkpoint_tag}] {name}\n{'='*60}")
        print(report)
        print(f"Accuracy: {acc:.4f} | Macro-F1 (3-class): {f1_all:.4f} | present-only: {f1_present:.4f}")
    safe = name.replace(" ", "_").lower()
    cm_path = os.path.join(CM_DIR, f"cm_{checkpoint_tag}_{safe}.png")
    save_confusion_matrix(y_true, y_pred, f"{checkpoint_tag} — {name}", cm_path, labels)
    return {
        "checkpoint": checkpoint_tag,
        "corpus": name,
        "n": int(len(y_true)),
        "accuracy": float(acc),
        "macro_f1_3class": float(f1_all),
        "macro_f1_present": float(f1_present),
        "gold_classes": present,
        "confusion_matrix_png": cm_path,
    }


CHECKPOINTS = [("phase1_best_val", best_path)]
if os.path.exists(phase2_final_path):
    CHECKPOINTS.append(("phase2_dann_final", phase2_final_path))
else:
    print(
        f"WARNING: {phase2_final_path} not found.\n"
        "Re-run the training cell once (after phase 2) to save phase-2 weights, then re-run this cell."
    )

all_eval_rows: List[Dict] = []
for ckpt_tag, ckpt_path in CHECKPOINTS:
    print(f"\n{'#'*60}\nOOD eval: {ckpt_tag}\n{'#'*60}")
    load_checkpoint(ckpt_path)
    for oname, odf in ood_eval.items():
        y_true = odf["label_id"].values
        y_pred = predict_texts(odf[TEXT_COL].tolist())
        all_eval_rows.append(eval_split(oname, y_true, y_pred, ckpt_tag, verbose=True))

comparison = pd.DataFrame(all_eval_rows)
print("\n" + "=" * 60)
print("OOD macro-F1 (3-class) — phase1 best val vs phase2 DANN final")
print("=" * 60)
pivot_f1 = comparison.pivot(index="corpus", columns="checkpoint", values="macro_f1_3class")
print(pivot_f1.to_string())
if {"phase1_best_val", "phase2_dann_final"}.issubset(pivot_f1.columns):
    pivot_f1["delta_dann_minus_valbest"] = (
        pivot_f1["phase2_dann_final"] - pivot_f1["phase1_best_val"]
    )
    print("\nDelta (phase2_dann_final - phase1_best_val):")
    print(pivot_f1["delta_dann_minus_valbest"].to_string())

mean_ood = comparison.groupby("checkpoint")["macro_f1_3class"].mean()
print("\nMean OOD macro-F1 across CoQA + RACE:")
print(mean_ood.to_string())
winner = mean_ood.idxmax()
print(f"\n→ Higher mean OOD macro-F1: {winner} ({mean_ood[winner]:.4f})")
print("  (Use this for domain-generalization; val-best may still be phase1_best_val.)")

results = {
    "config": {
        "model": MODEL_NAME,
        "dann": True,
        "label_col": LABEL_COL,
        "train_rows": int(len(df_train)),
        "val_rows": int(len(df_val)),
        "num_domains": num_domains,
        "coqa_in_train": False,
        "best_val_macro_f1": float(best_val_f1),
        "checkpoints": {tag: path for tag, path in CHECKPOINTS},
        "mean_ood_macro_f1": mean_ood.to_dict(),
        "ood_winner_by_mean_f1": winner,
        "history": history,
    },
    "evaluations": all_eval_rows,
    "pivot_macro_f1_3class": pivot_f1.reset_index().to_dict(orient="records"),
}

json_path = os.path.join(DRIVE_OUT_DIR, "eval_results.json")
csv_path = os.path.join(DRIVE_OUT_DIR, "eval_comparison.csv")
pivot_path = os.path.join(DRIVE_OUT_DIR, "eval_pivot_macro_f1.csv")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
comparison.to_csv(csv_path, index=False)
pivot_f1.reset_index().to_csv(pivot_path, index=False)
print(f"\nSaved → {json_path}")
print(f"Saved → {csv_path}")
print(f"Saved → {pivot_path}")